In [6]:
from xgboost import XGBRegressor
import requests
import pandas as pd
import sklearn
import requests
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
import re

def parse_wind_speed(speed):
    nums = re.findall(r"\d+", str(speed))
    nums = [float(x) for x in nums]
    if len(nums) == 0:
        return None
    return sum(nums) / len(nums)
    
#CURRENT

ERCOT_primary_key = '1b766749760c4fdcbc11936fbf9853eb'
day_ahead = "2024-09-01"
end_date = "2026-09-01"
auth_url = (
"https://ercotb2c.b2clogin.com/"
"ercotb2c.onmicrosoft.com/"
"B2C_1_PUBAPI-ROPC-FLOW/oauth2/v2.0/token")

Ercot_email = str(input("ERCOT EMAIL"))
ercot_password = str(input("ERCOT PAssword"))
payload = {"username": Ercot_email, "password": ercot_password,
           "grant_type": "password","scope": "openid fec253ea-0d06-4272-a5e6-b478baeecd70 offline_access",
           "client_id": "fec253ea-0d06-4272-a5e6-b478baeecd70","response_type": "id_token"}

token_response = requests.post(auth_url, data=payload)
print(token_response.status_code)
token_json = token_response.json()
access_token = token_json["access_token"]

ERCOTheaders = {"Authorization": f"Bearer {access_token}", "Ocp-Apim-Subscription-Key": ERCOT_primary_key}
ERCOTparams = {"settlementPoint": "HB_HOUSTON", "deliveryDateFrom": day_ahead}
ERCOTLparams = { "deliveryDateFrom": day_ahead}
DAM_Settlement_Point_Prices_url = "https://api.ercot.com/api/public-reports/np4-190-cd/dam_stlmnt_pnt_prices"
response = requests.get(DAM_Settlement_Point_Prices_url, headers=ERCOTheaders,params=ERCOTparams)

print(response.status_code)
ERCOT_data = response.json()
ERCOT_DAM_df = pd.DataFrame(ERCOT_data['data'])
ERCOT_DAM_df = ERCOT_DAM_df.rename(columns={0: "deliveryDate", 1: "hourEnding", 3: "settlementPointPrice"})


ERCOT_DAM_df["hour"] = (ERCOT_DAM_df["hourEnding"].str.split(":").str[0].astype(int))
ERCOT_DAM_df["timestamp"] = (pd.to_datetime(ERCOT_DAM_df["deliveryDate"])+ pd.to_timedelta(ERCOT_DAM_df["hour"] - 1, unit="h"))
ERCOT_DAM_df["hour"] = ERCOT_DAM_df["timestamp"].dt.hour
ERCOT_DAM_df = ERCOT_DAM_df.sort_values("timestamp")
ERCOT_DAM_df["dayofweek"] = ERCOT_DAM_df["timestamp"].dt.dayofweek
ERCOT_DAM_df["month"] = ERCOT_DAM_df["timestamp"].dt.month
ERCOT_DAM_df["weekend"] = ERCOT_DAM_df["dayofweek"].isin([5,6]).astype(int)
ERCOT_DAM_df["lag_24"] = ERCOT_DAM_df["settlementPointPrice"].shift(24)
ERCOT_DAM_df["lag_48"] = ERCOT_DAM_df["settlementPointPrice"].shift(48)
ERCOT_DAM_df["lag_168"] = ERCOT_DAM_df["settlementPointPrice"].shift(168)
ERCOT_DAM_df["roll24"] = (ERCOT_DAM_df["settlementPointPrice"].rolling(24).mean())
ERCOT_DAM_df["roll168"] = (ERCOT_DAM_df["settlementPointPrice"].rolling(168).mean())

#weather data 
url = "https://archive-api.open-meteo.com/v1/archive"

params = {"latitude": 29.98, "longitude": -95.36, "start_date": day_ahead, "end_date": end_date, "hourly": ["temperature_2m", "relative_humidity_2m",  "precipitation","wind_speed_10m" ]}
r = requests.get(url, params=params)
data = r.json()

hist_weather_df = pd.DataFrame(data["hourly"])
hist_weather_df["timestamp"] = pd.to_datetime(hist_weather_df["time"])

merged_ercot_hist_df = ERCOT_DAM_df.merge(hist_weather_df, on = "timestamp", how = "left")


#Train the Model
features = ["hour","dayofweek","weekend","month", "lag_24","lag_48","lag_168","roll24", "roll168","temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m"]
train = merged_ercot_hist_df.dropna().sort_values("timestamp")
X = train[features]
y = train["settlementPointPrice"]
split_idx = int(len(train) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

model = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    objective="reg:squarederror",
    random_state=42
)

model.fit(X_train, y_train)
preds = model.predict(X_test)
r2 = r2_score(y_test, preds)
mae = mean_absolute_error(y_test, preds)
print(f"XGBoost MAE: {mae:.3f}")
naive_preds = X_test["lag_24"]
naive_mae = mean_absolute_error(y_test, naive_preds)
print(f"Naive MAE: {naive_mae:.3f}")
improvement = (naive_mae - mae) / naive_mae * 100
print(f"Improvement over naive: {improvement:.2f}%")
# Feature importance
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)
print("\nFeature Importances:")
print(importance)

lat = 29.7604
lon = -95.3698

headersz = {"User-Agent": "ercot-price-model"}

r = requests.get(f"https://api.weather.gov/points/{lat},{lon}", headers=headersz)
print("NWS Status:", r.status_code)
data = r.json()
forecast_url = data["properties"]["forecastHourly"]
forecast = requests.get(forecast_url,headers=headersz).json()

weather_rows = []
for period in forecast["properties"]["periods"]:
    weather_rows.append({
        "timestamp": period["startTime"],
        "temperature": period["temperature"],
        "isDaytime": period["isDaytime"],
        "humidity": (
            period["relativeHumidity"]["value"]
            if period["relativeHumidity"] is not None
            else None
        ),
        "dewpoint": (
            period["dewpoint"]["value"]
            if period["dewpoint"] is not None
            else None
        ),
        "windSpeed": period["windSpeed"],
        "windDirection": period["windDirection"],
        "precipProb": (
            period["probabilityOfPrecipitation"]["value"]
            if period["probabilityOfPrecipitation"] is not None
            else 0)})

weather_df = pd.DataFrame(weather_rows)
weather_df["timestamp"] = (pd.to_datetime(weather_df["timestamp"]).dt.tz_localize(None))
weather_df["hour"] = weather_df["timestamp"].dt.hour
weather_df["dayofweek"] = weather_df["timestamp"].dt.dayofweek
weather_df["month"] = weather_df["timestamp"].dt.month
weather_df["weekend"] = (weather_df["dayofweek"].isin([5, 6]).astype(int))
weather_df = weather_df.rename(columns={
    "temperature": "temperature_2m",
    "humidity": "relative_humidity_2m",
    "precipProb": "precipitation",
    "windSpeed": "wind_speed_10m"})

weather_df["wind_speed_10m"] = (weather_df["wind_speed_10m"].apply(parse_wind_speed))
weather_df["relative_humidity_2m"] = (weather_df["relative_humidity_2m"].fillna(weather_df["relative_humidity_2m"].mean()))
weather_df["precipitation"] = (weather_df["precipitation"].fillna(0))


price_series = (ERCOT_DAM_df.set_index("timestamp")["settlementPointPrice"].copy())

future_predictions = []

for _, row in weather_df.iterrows():

    lag_24 = price_series.iloc[-24]
    lag_48 = price_series.iloc[-48]
    lag_168 = price_series.iloc[-168]
    roll24 = price_series.iloc[-24:].mean()
    roll168 = price_series.iloc[-168:].mean()

    X_future = pd.DataFrame([{
        "hour": row["hour"],
        "dayofweek": row["dayofweek"],
        "weekend": row["weekend"],
        "month": row["month"],
        "lag_24": lag_24,
        "lag_48": lag_48,
        "lag_168": lag_168,
        "roll24": roll24,
        "roll168": roll168,
        "temperature_2m": row["temperature_2m"],
        "relative_humidity_2m": row["relative_humidity_2m"],
        "precipitation": row["precipitation"],
        "wind_speed_10m": row["wind_speed_10m"]}])
    
    X_future = X_future[features]
    pred = model.predict(X_future)[0]
    future_predictions.append(pred)
    price_series.loc[row["timestamp"]] = pred


weather_df["predicted_price"] = future_predictions
print(weather_df[["timestamp", "predicted_price"]])

forecast_export = weather_df[["timestamp", "predicted_price"]].copy()
forecast_export["timestamp"] = (forecast_export["timestamp"].dt.tz_localize(None))
forecast_export.to_excel("HB_HOUSTON_FORECAST.xlsx", index=False)

print("\nForecast saved to:")
print("HB_HOUSTON_FORECAST.xlsx")
    

200
200
XGBoost MAE: 3.407
Naive MAE: 5.701
Improvement over naive: 40.23%

Feature Importances:
lag_24                  0.647992
relative_humidity_2m    0.056355
roll24                  0.053694
lag_168                 0.043832
hour                    0.042841
temperature_2m          0.036466
roll168                 0.035978
dayofweek               0.035258
lag_48                  0.031641
wind_speed_10m          0.012978
precipitation           0.002965
month                   0.000000
weekend                 0.000000
dtype: float32
NWS Status: 200
              timestamp  predicted_price
0   2026-09-15 14:00:00        41.865154
1   2026-09-15 15:00:00        42.012474
2   2026-09-15 16:00:00        41.644871
3   2026-09-15 17:00:00        41.644871
4   2026-09-15 18:00:00        41.726852
..                  ...              ...
151 2026-09-21 21:00:00       114.454086
152 2026-09-21 22:00:00        96.949844
153 2026-09-21 23:00:00        88.254593
154 2026-09-22 00:00:00        98

In [ ]:

import re

def parse_wind_speed(speed):
    nums = re.findall(r"\d+", str(speed))
    nums = [float(x) for x in nums]

    if len(nums) == 0:
        return None

    return sum(nums) / len(nums)

